# Pivots and Joins

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action='ignore',category=np.exceptions.VisibleDeprecationWarning)


In [ ]:
# polars
import polars as pl
from IPython.display import display


## Rows from lists

In [ ]:
# datascience
Table().with_columns('Numbers', [1, 2, 3])


In [ ]:
# polars
pl.DataFrame([
    pl.Series('Numbers', [1, 2, 3])
])


In [ ]:
# datascience
drinks = Table(['Drink', 'Cafe', 'Price'])
drinks


In [ ]:
# polars
drinks_pl = pl.DataFrame(schema={c: pl.Utf8 for c in ['Drink', 'Cafe', 'Price']})
drinks_pl


In [ ]:
# datascience
drinks = drinks.with_rows([
    ['Milk Tea', 'Asha', 5.5],
    ['Espresso', 'Strada',  1.75],
    ['Latte',    'Strada',  3.25],
    ['Espresso', "FSM",   2]
])
drinks


In [ ]:
# polars
drinks_pl = pl.DataFrame([
    ['Milk Tea', 'Asha', 5.5],
    ['Espresso', 'Strada',  1.75],
    ['Latte',    'Strada',  3.25],
    ['Espresso', "FSM",   2]
], schema=drinks_pl.columns, orient='row')
drinks_pl


## Cross-classification: grouping by two columns

Do right-handed people tend to sleep on their left side and left-handed people sleep on their right?

In [ ]:
# datascience
survey = Table.read_table('welcome_survey_fa26.csv')
survey.show(3)


In [ ]:
# polars
survey_pl = pl.read_csv('welcome_survey_fa26.csv')
survey_pl.head(3)


In [ ]:
# datascience
survey.group(['Handedness', 'Sleep Position']).show()


In [ ]:
# polars
with pl.Config(tbl_rows=-1):
    display(survey_pl.group_by(['Handedness', 'Sleep Position']).len(name='count').sort(['Handedness', 'Sleep Position']))


In [ ]:
# datascience
survey.group(['Handedness', 'Sleep Position'], np.average).show()


In [ ]:
# polars
with pl.Config(tbl_rows=-1):
    display((lambda _df: _df.group_by(['Handedness', 'Sleep Position']).agg([getattr(pl.col(c), 'mean')().alias(f'{c} average') for c, t in _df.schema.items() if c not in (['Handedness', 'Sleep Position'] if isinstance(['Handedness', 'Sleep Position'], list) else [['Handedness', 'Sleep Position']]) and (t.is_numeric())]).sort(['Handedness', 'Sleep Position']))(survey_pl))


## Pivot Tables

In [ ]:
# datascience
survey.group('Handedness')


In [ ]:
# polars
survey_pl.group_by('Handedness').len(name='count').sort('Handedness')


In [ ]:
# datascience
survey.pivot('Sleep Position', 'Handedness')


In [ ]:
# polars
survey_pl.pivot('Sleep Position', index='Handedness', aggregate_function='len').sort('Handedness')


**DISCUSS**: Why do the group and pivot tables have different numbers of entries?

In [ ]:
# datascience
survey.pivot('Sleep Position', 
             'Handedness', 
             values='Sleep Hours', 
             collect=np.average)


In [ ]:
# polars
survey_pl.pivot('Sleep Position', index='Handedness', values='Sleep Hours', aggregate_function='mean').sort('Handedness')


**DISCUSS**: In the above pivot table, what does one number mean?

## Discussion Questions ##

In [ ]:
# datascience
# From the CORGIS Dataset Project
# By Austin Cory Bart acbart@vt.edu
# Version 2.0.0, created 3/22/2016
# https://corgis-edu.github.io/corgis/csv/skyscrapers/

sky = Table.read_table('skyscrapers.csv')
sky = (sky.with_column('age', 2022 - sky.column('completed'))
          .drop('completed'))
sky.show(3)


In [ ]:
# polars
# From the CORGIS Dataset Project
# By Austin Cory Bart acbart@vt.edu
# Version 2.0.0, created 3/22/2016
# https://corgis-edu.github.io/corgis/csv/skyscrapers/

sky_pl = pl.read_csv('skyscrapers.csv')
sky_pl = (sky_pl.with_columns((2022 - sky_pl.get_column('completed')).alias('age'))
          .drop('completed'))
sky_pl.head(3)


### 1. For each city, what’s the height of the tallest building for each material?

In [ ]:
# YOUR CODE HERE



### 2. For each city, what’s the height difference between the tallest steel building and the tallest concrete building?

In [ ]:
# YOUR CODE HERE












---

#### **WARNING: ANSWERS BELOW**

In [ ]:
# datascience
sky.select('material', 'city', 'height').group(['city', 'material'], collect=max)


In [ ]:
# polars
(lambda _df: _df.group_by(['city', 'material']).agg([pl.col(c).max().alias(f'{c} max') for c, t in _df.schema.items() if c not in (['city', 'material'] if isinstance(['city', 'material'], list) else [['city', 'material']]) and (True)]).sort(['city', 'material']))(sky_pl.select('material', 'city', 'height'))


In [ ]:
# datascience
tallest = sky.pivot('material', 'city', values='height', collect=max)
tallest.show()


In [ ]:
# polars
tallest_pl = sky_pl.pivot('material', index='city', values='height', aggregate_function='max').sort('city')
with pl.Config(tbl_rows=-1):
    display(tallest_pl)


In [ ]:
# datascience
diffs = Table().with_columns(
    'city', tallest.column('city'),
    'difference', abs(tallest.column('steel') - tallest.column('concrete'))
)
diffs


In [ ]:
# polars
diffs_pl = pl.DataFrame([
    (tallest_pl.get_column('city')).alias('city'),
    (abs(tallest_pl.get_column('steel') - tallest_pl.get_column('concrete'))).alias('difference')
])
diffs_pl


In [ ]:
# datascience
diffs.sort('difference', descending=True)


In [ ]:
# polars
diffs_pl.sort('difference', descending=True)


### 3. Generate a table of the names of the oldest buildings for each material for each city.

In [ ]:
# datascience
# Hint: You can use sort to find the name of the oldest building in the dataset
sky.sort('age', descending=True).column('name').item(0)

# YOUR CODE HERE


In [ ]:
# polars
# Hint: You can use sort to find the name of the oldest building in the dataset
sky_pl.sort('age', descending=True).get_column('name').item(0)

# YOUR CODE HERE


---

#### **WARNING: ANSWERS BELOW**

In [ ]:
# datascience
def first(s):
    "Return the first element in an array."
    return s.item(0)

(sky
 .sort('age', descending=True)
 .pivot('material', 'city', 'name', first)
)#.where('city', 'San Francisco')


In [ ]:
# polars
def first_pl(s):
    "Return the first element in an array."
    return s.item(0)

(sky_pl
 .sort('age', descending=True).pivot('material', index='city', values='name', aggregate_function='first').sort('city')
)#.where('city', 'San Francisco')


## Joins ##

In [ ]:
drinks

In [ ]:
# datascience
discounts = Table().with_columns(
    'Coupon % off', make_array(10, 25, 5),
    'Location', make_array('Asha', 'Strada', 'Asha')
)
discounts


In [ ]:
# polars
discounts_pl = pl.DataFrame([
    (pl.Series('values', [10, 25, 5])).alias('Coupon % off'),
    (pl.Series('values', ['Asha', 'Strada', 'Asha'])).alias('Location')
])
discounts_pl


In [ ]:
# datascience
combined = drinks.join('Cafe', discounts, 'Location')
combined


In [ ]:
# polars
combined_pl = drinks_pl.sort('Cafe').join(discounts_pl.sort('Location'), left_on='Cafe', right_on='Location')
combined_pl


In [ ]:
# datascience
discounted_frac = 1 - combined.column('Coupon % off') / 100
combined.with_column(
    'Discounted Price', 
    combined.column('Price') * discounted_frac
)


In [ ]:
# polars
discounted_frac_pl = 1 - combined_pl.get_column('Coupon % off') / 100
combined_pl.with_columns((combined_pl.get_column('Price') * discounted_frac_pl).alias('Discounted Price'))


In [ ]:
# datascience
drinks.join('Cafe', drinks, 'Cafe')


In [ ]:
# polars
drinks_pl.sort('Cafe').join(drinks_pl.sort('Cafe'), on='Cafe')
